# 03. Inheritance

**Tier:** Intermediate. **Why you'd use this:** A counter should gain a step that adds one more than the caller asked. A subclass can override `add` and still reuse the parent's storage through `super()`.

**After this lesson you can:** override a method; call `super()`; read `__mro__` to see lookup order.
About 25 minutes. **Uses:** [Classes](01-classes-lesson.ipynb)


## Quick review

1. What does `@dataclass` generate for equality?
2. Why does `frozen=True` help with hashing?

<details><summary>Answers</summary>

Field-value `__eq__`. Immutability makes a stable hash possible.
</details>


## The idea

A subclass reuses a base class and can override methods. `super()` continues along the method resolution order so cooperative overrides can each run. Print `__mro__` when you need to see that order. Inheritance is for "is a specialized kind of," not for grabbing unrelated helpers.

<details><summary>Explain more</summary>

`Step.add` can call `super().add(n + 1)` to reuse `Counter.add` after adjusting the argument.
</details>


## See it work

Specialize a counter with an extra step.


In [1]:
class Counter:
    def __init__(self):
        self.value = 0

    def add(self, n):
        self.value += n
        return self.value


class Step(Counter):
    def add(self, n):
        # 1. Continue to the next class in the MRO.
        return super().add(n + 1)


print(Step().add(1))
print(Step.__mro__)


2
(<class '__main__.Step'>, <class '__main__.Counter'>, <class 'object'>)


## Check yourself

Answer these before you run the cell.

1. Which class's `add` runs first for `Step().add(1)`?
2. What does `super()` mean in a diamond hierarchy?


In [2]:
class Counter:
    def __init__(self):
        self.value = 0

    def add(self, n):
        self.value += n
        return self.value


class Step(Counter):
    def add(self, n):
        return super().add(n + 1)


print(Step().add(0))
print([cls.__name__ for cls in Step.__mro__])


1
['Step', 'Counter', 'object']


<details><summary>Answers</summary>

`Step.add` runs first, then `Counter.add` through `super`. Continue to the next class in the MRO, not a hard-coded parent only.
</details>


## Why it works this way

Inheritance puts base classes in a method resolution order, the MRO. Attribute lookup walks that order after the instance dict. `super()` means "continue along the MRO from here," not "call my parent only." In a diamond, cooperative `super()` lets each class run once when every override forwards with `super()`. Printing `Class.__mro__` shows the order the language will use. Deep dive: [Attribute lookup](../../internals/08-attributes/01-lookup-lesson.ipynb).

Subclassing reuses and specializes behaviour. `Step.add` can call `super().add(...)` to reuse `Counter.add` while changing the argument. Without `super()`, you either duplicate code or hard-code a parent name that breaks when the hierarchy changes. The design of `super` in modern Python is cooperative multiple inheritance. Even with one parent, using `super()` keeps that habit.

Cost of a method call through `super` is still a normal call after MRO resolution. The conceptual cost is understanding who runs next. Prefer composition when reuse is only a helper object. Prefer inheritance when the subclass truly is a specialized version of the base and will be used wherever the base is expected.

Evidence: `Step.__mro__` lists `Step`, `Counter`, `object`, and `Step().add(1)` returns a value that shows the extra `+ 1` happened before the base add. A diamond with three `super` prints shows each class once when cooperation holds.


When an override forgets `super`, the chain stops and later classes in the MRO never see the call. That is fine for a full replacement and wrong when each layer should contribute. Keep overrides small. Forward with `super` unless you are intentionally cutting the chain. Reading `__mro__` once when you design the hierarchy saves a debugging hour later. Composition stays available: hold a `Counter` inside another object when "is-a" feels forced.


In [3]:
class Counter:
    def add(self, n):
        return n


class Step(Counter):
    def add(self, n):
        return super().add(n + 1)


print(Step().add(1))
print([cls.__name__ for cls in Step.__mro__])


class A:
    def show(self):
        return "A"


class B(A):
    def show(self):
        return "B" + super().show()


class C(A):
    def show(self):
        return "C" + super().show()


class D(B, C):
    def show(self):
        return "D" + super().show()


print(D().show())
print([cls.__name__ for cls in D.__mro__])


2
['Step', 'Counter', 'object']
DBCA
['D', 'B', 'C', 'A', 'object']


## When to use it

Use inheritance when the subclass is a true specialization. Use `super()` in overrides so cooperation stays possible. Print `__mro__` when lookup surprises you. Prefer composition when you only need a helper object. Don't inherit just to reuse one unrelated method.


## Common mistake

Hard-coding the parent class name instead of `super()`, then inserting another base later and wondering why the new class never runs.


In [4]:
class Counter:
    def add(self, n):
        return n


class Step(Counter):
    def add(self, n):
        return Counter.add(self, n + 1)


print(Step().add(1))


2


Call `super().add(...)` so the MRO decides who runs next.


In [5]:
class Counter:
    def add(self, n):
        return n


class Step(Counter):
    def add(self, n):
        return super().add(n + 1)


print(Step().add(1))


2


## Try it

Add a `Double(Counter)` that doubles `n` before adding. In a class `Both(Step, Double)`, what does `Both.__mro__` print, and who runs first?


## Recap

- Subclasses override methods and may call `super()`.
- `super()` follows the MRO, not a single hard-coded parent.
- Print `__mro__` to see lookup order.

**Next:** [practice](03-inheritance-practice.ipynb)
